# Cassette fronts: R35 reviewed-annotation comparison

**Choose an L4 or A100 GPU in Colab, select Run all, and return `baomai-front-review-r35-results.zip`.**

One question: **do the accepted R35 annotation corrections improve cassette-front detection and corner localization?** The reference uses accepted R34 training labels. The candidate changes **5 rows across 3 training photographs**, with unchanged object, class and corner-availability inventories. Every other training setting is shared.

Both arms use the unmodified native Pose26 loss and start independently from original epoch 173. Each receives 137 training images, including 13 empty scenes, once per epoch: **30 epochs, 1,050 batches, 4,110 image presentations and 78 optimizer updates**, seed 46, batch 4, image size 1280, FP32 and AdamW. Final-epoch selection is fixed.

The notebook loads a published immutable source commit. Labels and review records live in the repository, keeping this notebook small. Five code cells prepare the workspace, restore inputs, verify the runtime, run a discarded four-batch trial and the matched pair, then package the result. Compatible completed runs are reused and backed up to Drive when available.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

SOURCE_REVISION = 'bfe57e5ceb95ffa1d94ee56880010052ed84c86d'
PROTOCOL_PATH = 'experiments/front-review-2026-11-25-r35.json'
PROTOCOL_SHA256 = 'b74cf472f42380f7f8c373d9ad9fda1e88560d16ca2fb64d079741d403c5a80d'
EXPERIMENT_ID = 'front-review-r35'
ROLE_RECIPES = {'reference': 'reviewed-incremental-fronts-seed46-v1', 'candidate': 'reviewed-incremental-fronts-seed46-v1'}
RECIPE_SHA256 = 'f18e0a813a1057e1b3ab3a44ea3ce9b2a3e7673cc2bf04ca09b394bfdd8c51b8'
RUN_BASE = (Path("/content") if Path("/content").exists() else Path.cwd()) / "baomai-front-review-r35"
ROLES = ('reference', 'candidate')

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Use the final handoff notebook with its full source commit pinned."
assert re.fullmatch(r"[0-9a-f]{64}", PROTOCOL_SHA256), "Use the final handoff notebook with its protocol hash pinned."
assert sys.version_info >= (3, 11), "Python3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

workspace_identity = {"source_revision": SOURCE_REVISION, "protocol_sha256": PROTOCOL_SHA256}
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    assert RUN_BASE.is_dir() and not RUN_BASE.is_symlink(), "Choose a fresh nonsymlink workspace."
    marker = RUN_BASE / "workspace-identity.json"
    assert marker.is_file() and json.loads(marker.read_text()) == workspace_identity, "Existing workspace belongs to different pins; preserve it and choose a new RUN_BASE."
else:
    RUN_BASE.mkdir(parents=True)
    (RUN_BASE / "workspace-identity.json").write_text(json.dumps(workspace_identity, indent=2) + "\n")
REPO = RUN_BASE / "dl"
DRIVE_ROOT = None
try:
    from google.colab import drive
except ImportError:
    print("Drive is unavailable here; preserve completed bundles and the ZIP from the local workspace.")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/baomai-front-review-r35-20261125")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)

def safe_member(name):
    p = Path(name)
    assert isinstance(name, str) and name and not p.is_absolute() and ".." not in p.parts
    assert "\\" not in name and ":" not in name and p.as_posix() == name
    return p

def artifact_inventory(path):
    path = Path(path)
    if path.is_file():
        names = [path.name]
        base = path.parent
    else:
        base = path
        receipt = json.loads((path / "training-receipt.json").read_text())
        names = ["training-receipt.json", "staging-manifest.json", receipt["log"]["path"]]
        if (path / "checkpoint-manifest.json").is_file():
            manifest = json.loads((path / "checkpoint-manifest.json").read_text())
            names += ["checkpoint-manifest.json", manifest["checkpoint"]["path"], receipt["results"]["path"]]
    assert len(names) == len(set(names))
    return [{"path": name, "sha256": sha256(base / safe_member(name)), "size_bytes": (base / name).stat().st_size}
            for name in sorted(names)]

def validate_copy(path, marker):
    base = path if marker["directory"] else path.parent
    for entry in marker["files"]:
        target = base / safe_member(entry["path"])
        assert target.is_file() and not target.is_symlink(), f"Missing backup artifact: {target}"
        assert target.stat().st_size == entry["size_bytes"] and sha256(target) == entry["sha256"], f"Backup hash mismatch: {target}"

def save_to_drive(path):
    if DRIVE_ROOT is None:
        return
    path = Path(path)
    relative = path.relative_to(RUN_BASE).as_posix()
    if relative == "resource-trial":
        relative = "resource-trials/" + sha256(path / "training-receipt.json")[:20]
    destination = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(destination) + ".complete")
    marker = {**workspace_identity, "relative_path": relative, "directory": path.is_dir(), "files": artifact_inventory(path)}
    if marker_path.is_file():
        assert json.loads(marker_path.read_text()) == marker, f"Drive already has different completed evidence: {destination}"
        validate_copy(destination, marker)
        print("Verified existing Drive backup:", destination)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Only an incomplete copy in this notebook's exact destination is replaced.
    if destination.is_dir():
        shutil.rmtree(destination)
    elif destination.exists():
        destination.unlink()
    if path.is_dir():
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path / entry["path"], target)
    else:
        shutil.copyfile(path, destination)
    validate_copy(destination, marker)
    marker_path.write_text(json.dumps(marker, indent=2) + "\n")
    print("Saved verified completed artifacts:", destination, flush=True)

def restore_from_drive(relative):
    if DRIVE_ROOT is None:
        return None
    source = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(source) + ".complete")
    if not marker_path.is_file():
        return None
    marker = json.loads(marker_path.read_text())
    assert all(marker.get(k) == v for k, v in workspace_identity.items()), "Drive completion belongs to different source/protocol pins."
    assert marker["relative_path"] == relative
    validate_copy(source, marker)
    destination = RUN_BASE / safe_member(relative)
    if destination.exists() or destination.is_symlink():
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if marker["directory"]:
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / entry["path"], target)
    else:
        assert len(marker["files"]) == 1 and marker["files"][0]["path"] == source.name
        shutil.copyfile(source, destination)
    validate_copy(destination, marker)
    print("Restored completed evidence:", destination, flush=True)
    return destination

print("Workspace:", RUN_BASE)

SOURCE_URL = "https://github.com/Siuuuers/dl.git"

def assemble_source(repo, origin=SOURCE_URL):
    repo = Path(repo)
    assert not repo.is_symlink(), f"Source folder is a symlink: {repo}. Choose a fresh RUN_BASE."
    assert not repo.exists() or repo.is_dir(), f"Preserve the existing file at {repo} and choose a fresh RUN_BASE."
    git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
    if not repo.exists():
        run(["git", "init", repo], env=git_env)
        run(["git", "remote", "add", "origin", origin], cwd=repo)
        run(["git", "fetch", "--depth=1", "origin", SOURCE_REVISION], cwd=repo, env=git_env)
        run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=repo, env=git_env)
    else:
        probe = subprocess.run(["git", "rev-parse", "--verify", "HEAD"], cwd=repo,
                               capture_output=True, text=True)
        assert probe.returncode == 0, (f"Source checkout is incomplete at {repo}. Preserve or rename only "
            "that source folder, then rerun. Keep the run and Drive folders intact.")
        assert probe.stdout.strip() == SOURCE_REVISION, (
            "Existing source belongs to different pins; preserve it and choose a fresh RUN_BASE.")
    assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() == SOURCE_REVISION
    assert sha256(repo / PROTOCOL_PATH) == PROTOCOL_SHA256
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), (
        "Existing source has local changes. Preserve or rename only the source folder; keep completed runs intact.")
    print("Verified published R35 source:", SOURCE_REVISION)
    return SOURCE_REVISION


## 1. Restore source and original inputs

The exact source commit, protocol hash, 157 original image hashes and unchanged epoch-173 initializer are checked. Existing incompatible or incomplete folders are preserved for inspection. The notebook does not embed source files, labels, photos or weights.


In [ ]:
def verify_original_inputs(repo):
    repo = Path(repo)
    manifest = json.loads((repo / "data/manifest.json").read_text())
    protocol = json.loads((repo / PROTOCOL_PATH).read_text())
    assert len(manifest["records"]) == 157
    for record in manifest["records"]:
        path = repo / "data" / safe_member(record["image"])
        assert sha256(path) == record["image_sha256"], f"Original image differs: {record['id']}"
    assert manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
    assert sha256(repo / manifest["model_checkpoint"]) == protocol["initial_checkpoint_sha256"]
    assert subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo).returncode == 0
    # Refresh only verified original LFS paths. Their pointer objects and the
    # committed source must stay identical; no changed inputs are committed.
    run(["git", "add", "--renormalize", "--", "data/images", manifest["model_checkpoint"]], cwd=repo)
    assert subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo).returncode == 0, (
        "Restored originals changed indexed pointer objects. Preserve this source folder for inspection.")
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), (
        "Source must remain clean after verified original-input restoration.")
    print("Verified 157 original images and the unchanged epoch173 initializer.")
    return {"images_verified": 157, "initializer_sha256": protocol["initial_checkpoint_sha256"]}

lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
assemble_source(REPO)

# Original images and fixed initializer (published data only).
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"], cwd=REPO, text=True).splitlines())
assert len(expected_images) == 157 and tracked_images == expected_images
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt", "--exclude="], cwd=REPO)
verify_original_inputs(REPO)

## 2. Verify the exact training change

Only the accepted corrections in 5 eligible training rows differ. All 137 training photographs remain, with 3,784 objects and 13 empty scenes. Held photograph 015 and the 31 excluded old-series photographs retain their exact previous supervision in both arms. The original trainer-validation split remains 20 images with 664 objects.

The R35 review includes development-only corrections and removal of preserved R34 `new_000009` rows 56 and 57. These changes never enter training or trainer validation. Review IDs retain the original R34 row identities; R35 TXT positions are separately mapped after deletion. Frozen 666 and historical 663 evaluation references remain unchanged. R34 and R35 development reviews are separately identified exploratory references. A corrected reference is not a measured model improvement.


In [ ]:
import torch
print("Runtime:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (completed-pair recovery only)")
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")), encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__)"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Limits: 600 seconds for the four-batch trial; 21600 seconds for each complete run.")


sys.dont_write_bytecode = True
sys.path.insert(0, str(REPO / "scripts"))
from evaluate_pose import preflight_dataset
from stage_pose_training import source_state
from partial_box_loss_r31 import verify_native_sources
from native_front_review_r35 import load_protocol, source_identity, verify_run, verify_trial, package_pair
verify_native_sources()  # Verify pinned native code; no custom criterion is attached.
CONTEXT = load_protocol(REPO, PROTOCOL_PATH)
assert CONTEXT["sha256"] == PROTOCOL_SHA256
document = CONTEXT["document"]
assert document["experiment_id"] == EXPERIMENT_ID
assert document["recipe_sha256"] == RECIPE_SHA256
assert all(document["recipe_id"] == ROLE_RECIPES[role] for role in ROLES)
assert document["execution"]["loss"] == "unmodified_native_Pose26"
PREFLIGHT = preflight_dataset(REPO, require_weights=True)
assert PREFLIGHT["model_sha256"] == document["initial_checkpoint_sha256"]
STATE = source_state(REPO, REPO / document["candidate"]["path"])
for role in ROLES:
    assert STATE["counts"][role] == document["roles"][role]["counts"]
    assert STATE["counts"][role]["train"]["images"] == 137
    assert STATE["counts"][role]["train"]["empty_images"] == 13
    assert STATE["counts"][role]["val"]["objects"] == 664
DELTA = CONTEXT["candidate_verification"]["training_input_delta"]["report"]
assert DELTA["summary"]["changed_rows"] > 0
assert DELTA["summary"]["classes_preserved"] is True
DELTA_PATH = RUN_BASE / "training-input-delta.json"
delta_bytes = (REPO / document["training_input_delta"]["path"]).read_bytes()
assert not DELTA_PATH.exists() or DELTA_PATH.read_bytes() == delta_bytes
DELTA_PATH.write_bytes(delta_bytes)
save_to_drive(DELTA_PATH)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified training change:", DELTA["summary"])
print("Reference uses accepted R34 labels; candidate uses the exact eligible R35 corrections.")
print("Native Pose26 loss in both arms. Missing landmarks remain unavailable supervision.")
print("Each arm: seed 46; 137 images once per epoch; 30 epochs; 4110 image presentations; 1050 batches; 78 optimizer updates.")
print("Batch 4, image size 1280, FP32, AdamW, initial learning rate 0.0001. Resource-trial weights are discarded.")


## 3. Run the matched native-loss pair

Use one GPU runtime for both arms. The four-batch resource trial checks real loading, gradients, parameter updates and memory; its weights are discarded. Each full arm then starts afresh from original epoch 173. This experiment changes annotation supervision only.


In [ ]:
def assert_finished(output, role, *, trial=False):
    bundle = (verify_trial(REPO, output, role, CONTEXT) if trial
              else verify_run(REPO, output, role, CONTEXT))
    receipt = bundle["receipt"] if "receipt" in bundle else bundle
    assert receipt["source"] == source_identity(REPO)
    assert receipt["runtime"]["device"].startswith("cuda:")
    return receipt

def assert_current_runtime(receipt, current, label):
    expected = receipt["runtime"]
    assert all(expected[k] == v for k, v in current.items() if k != "cudnn_version"), f"Current runtime differs from {label}; select a matching runtime before full training."
    assert expected["backend"]["cudnn_version"] == current["cudnn_version"], f"Current cuDNN differs from {label}."

def train_one(role, *, trial=False):
    folder = "resource-trial" if trial else role
    output = RUN_BASE / folder
    if output.exists() or output.is_symlink():
        try:
            receipt = assert_finished(output, role, trial=trial)
        except Exception as error:
            raise FileExistsError(f"Preserve incomplete or incompatible run {output}. Archive only this incomplete folder, then retry with the same workspace. {error}") from error
        print("Reusing verified completed run:", folder, flush=True)
        save_to_drive(output)
        return output, receipt
    command = [sys.executable, "-B", "scripts/train_pose_comparison.py", "--root", str(REPO),
               "--candidate", document["candidate"]["path"], "--recipe", ROLE_RECIPES[role], "--trial" if trial else "--train", "--arm", role,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting", folder, "with its verified label arm and native loss. Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}
    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass
    started, last_notice = time.monotonic(), -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}: {elapsed:.0f}s; {progress.get('status')}; epochs {len(progress.get('completed_epochs', []))}/30; batches {progress.get('batches_completed', 0)}; updates {progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}: verifying and staging inputs ({elapsed:.0f}s)", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed; preserve its log and receipt. No partial run is accepted.")
    except BaseException:
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = assert_finished(output, role, trial=trial)
    save_to_drive(output)
    return output, receipt

COMPLETED = {}
for role in ROLES:
    restore_from_drive(role)
    output = RUN_BASE / role
    if output.exists() or output.is_symlink():
        _, COMPLETED[role] = train_one(role)

TRIAL_RUNTIME = None
if len(COMPLETED) == 2:
    assert COMPLETED["reference"]["runtime"] == COMPLETED["candidate"]["runtime"]
    print("Both roles are complete; no training will repeat.")
else:
    assert HOST_RSS_LIMIT_MIB >= 6000, "Choose a runtime with more free host RAM."
    probe = r"""
import importlib.metadata as m, json, platform, torch
assert torch.cuda.is_available(), "Select a CUDA GPU runtime before starting an incomplete pair."
assert (torch.ones(1, device='cuda:0') * 2).item() == 2
print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,
'ultralytics':m.version('ultralytics'),'torch_cuda':torch.version.cuda,'platform':platform.platform(),
'device':'cuda:0','device_name':torch.cuda.get_device_name(0),'device_capability':list(torch.cuda.get_device_capability(0)),
'packages':{n:m.version(n) for n in ('torchvision','numpy','Pillow','opencv-python','psutil')},
'cudnn_version':torch.backends.cudnn.version()}))
"""
    current = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True, cwd=REPO))
    for role, receipt in COMPLETED.items():
        assert_current_runtime(receipt, current, f"completed {role}")
    TRIAL, trial_receipt = train_one("candidate", trial=True)
    assert_current_runtime(trial_receipt, current, "resource trial")
    TRIAL_RUNTIME = trial_receipt["runtime"]
    for role, receipt in COMPLETED.items():
        assert receipt["runtime"] == TRIAL_RUNTIME, f"Trial differs from completed {role}; no full role started."

REFERENCE_RUN, reference_receipt = train_one("reference")
if TRIAL_RUNTIME is not None:
    assert reference_receipt["runtime"] == TRIAL_RUNTIME, "Control runtime differs from trial."
CANDIDATE_RUN, candidate_receipt = train_one("candidate")
assert reference_receipt["runtime"] == candidate_receipt["runtime"], "The two roles use different runtimes."
assert reference_receipt["training_labels"]["label_manifest"] == document["control"]["manifest"]
assert candidate_receipt["training_labels"]["label_manifest"] == document["candidate"]["path"]
assert reference_receipt["training_labels"]["training_protocol"] == candidate_receipt["training_labels"]["training_protocol"]
assert [s["image_ids"] for s in reference_receipt["batch_samples"]] == [s["image_ids"] for s in candidate_receipt["batch_samples"]]
assert reference_receipt["optimizer_steps"] == candidate_receipt["optimizer_steps"] == 78
assert reference_receipt["batches_completed"] == candidate_receipt["batches_completed"] == 1050
assert reference_receipt["images_processed"] == candidate_receipt["images_processed"] == 4110
print("Both roles completed with the same training budget. Package the verified artifacts next.")

## 4. Return the verified result ZIP

Return `baomai-front-review-r35-results.zip` with both final checkpoints, raw logs, receipts, protocol and exact training delta. Packaging verifies artifact bytes, supervision and runtime equality. A completed compatible ZIP can be verified and downloaded again without retraining.


In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-front-review-r35-results.zip"
restore_from_drive(RESULTS_ZIP.relative_to(RUN_BASE).as_posix())
result = package_pair(REPO, REFERENCE_RUN, CANDIDATE_RUN, RESULTS_ZIP,
                      verify_existing=RESULTS_ZIP.exists(), protocol=PROTOCOL_PATH)
print(json.dumps(result, indent=2))
save_to_drive(RESULTS_ZIP)
try:
    from google.colab import files
except ImportError:
    print("Completed results:", RESULTS_ZIP)
else:
    files.download(str(RESULTS_ZIP))

## How to interpret the returned run

Keep inference fixed: box confidence 0.13, endpoint confidence 0.50, image size 1280, one-to-many output and NMS IoU 0.70. Matching IoU remains 0.50. Report frozen 666 and historical 663 separately, with R34/R35 development references clearly marked exploratory.

Prioritize missed visible fronts, especially fronts with zero or one locatable corner. Report unmatched outputs and their duplicate subset, empty-scene outputs, and both-corner successes at 5/10 original-image pixels alongside misses. Keep missed eligible objects in success denominators. A box-localization mismatch can create both an unmatched output and a missed reference; it is not automatically a completely absent prediction.

PR35 remains selected. This notebook does not promote a model, change inference thresholds, add hidden annotation points or claim reliability on unseen camera scenes. A hidden-corner estimate at inference has no coordinate truth when the annotation is unavailable.
